# 3.40 — Probability Calibration

Probability calibration asks whether a model's predicted probabilities behave like honest frequencies: among cases scored near 0.70, do about 70% actually happen? In this lesson, you will build calibration checks from first principles, bin predictions by hand, compute calibration gaps and proper losses, and see why the model we carry forward is chosen by the full decision score rather than the prettiest raw training number.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build probability calibration one idea at a time. Run each cell in order and read the printed intermediate values — every average, bin, gap, and score is visible. This walkthrough is self-contained, uses only NumPy and Matplotlib, and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, masks, binning, and numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for the small simulations.

### 1. Probabilities are frequency contracts

A predicted probability is not a class label. A score of 0.80 means a contract: if we collect many similar predictions near 0.80, roughly 80% of them should have label 1. Calibration is therefore checked over groups of examples, not by asking whether a single 0.80 prediction was right or wrong.

In [ ]:
p_w = np.array([0.10, 0.20, 0.30, 0.40, 0.60, 0.70, 0.80, 0.90])  # predicted probabilities.
y_w = np.array([0,    0,    1,    1,    1,    1,    1,    1])      # realized binary outcomes.
print("predicted probabilities:", p_w)
print("observed labels:        ", y_w)
print("overall predicted mean:", round(float(np.mean(p_w)), 3))
print("overall event rate:    ", round(float(np.mean(y_w)), 3))

▶ What you'll see: the average predicted probability is 0.500, while the observed event rate is 0.750, so this tiny sample is underconfident overall.

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(p_w, y_w, s=80, color="teal")
plt.plot([0, 1], [0, 1], "k--", label="perfect long-run frequency")
plt.xlabel("predicted probability")
plt.ylabel("observed label")
plt.title("1: single labels are noisy; calibration is a group property")
plt.legend()
plt.show()

▶ What you'll see: labels are only 0 or 1, so individual points cannot lie smoothly on the diagonal; the diagonal becomes meaningful after averaging groups.

*Why it's done this way:* A Bernoulli outcome has variance $p(1-p)$, so one case scored 0.8 can still be 0 without making the score wrong. Calibration replaces impossible single-case judgment with a frequency check: average the labels in a comparable group and compare that empirical frequency to the average predicted probability.

### 2. Binning turns probabilities into calibration gaps

The core lesson formula is `calibration gap = |Pr(Y=1 | p in bin) - avg(p in bin)|`. We choose probability bins, collect examples in each bin, compute the average predicted probability and the observed event frequency, and then take the absolute difference.

In [ ]:
bins_w = np.array([0.0, 0.5, 0.75, 1.0])  # low, medium, high probability regions.
bin_id_w = np.digitize(p_w, bins_w[1:-1], right=True)  # assign each prediction to a bin index.
print("bin ids:", bin_id_w)
print("bin edges:", bins_w)

▶ What you'll see: the eight probabilities are split into three regions: up to 0.5, 0.5–0.75, and above 0.75.

In [ ]:
avg_p_w, freq_w, gap_w, count_w = [], [], [], []
for b_w in range(len(bins_w) - 1):
    mask_w = bin_id_w == b_w
    avg_p_w.append(float(np.mean(p_w[mask_w])))
    freq_w.append(float(np.mean(y_w[mask_w])))
    gap_w.append(abs(freq_w[-1] - avg_p_w[-1]))
    count_w.append(int(np.sum(mask_w)))
print("bin counts:", count_w)
print("avg p per bin:", np.round(avg_p_w, 3))
print("event freq per bin:", np.round(freq_w, 3))
print("absolute gaps:", np.round(gap_w, 3))
assert np.allclose(np.round(gap_w, 3), [0.25, 0.35, 0.15])

▶ What you'll see: the medium bin has the largest gap (0.350), because its average score is 0.650 but every observed label in that bin is 1.

In [ ]:
plt.figure(figsize=(4.5, 3.5))
plt.plot([0, 1], [0, 1], "k--", label="perfect calibration")
plt.scatter(avg_p_w, freq_w, s=np.array(count_w) * 80, color="purple", alpha=0.8)
for x_w, ybin_w, g_w in zip(avg_p_w, freq_w, gap_w):
    plt.plot([x_w, x_w], [x_w, ybin_w], color="gray", linewidth=2)
plt.xlabel("average predicted probability")
plt.ylabel("observed event frequency")
plt.title("2: reliability diagram gaps")
plt.legend()
plt.show()

▶ What you'll see: vertical gray segments show calibration error in each bin; points above the diagonal mean events happened more often than predicted.

*Why it's done this way:* Bins trade resolution for statistical stability. A bin's empirical frequency estimates $\Pr(Y=1\mid \hat p\in\text{bin})$, while its average prediction estimates what the model promised. Their difference is the local violation of the probability contract.

### 3. Expected calibration error summarizes many bins

A single bin gap is useful, but model comparison needs one number. Expected calibration error (ECE) weights each bin's absolute gap by the fraction of examples in that bin, so large bins matter more than tiny bins.

In [ ]:
weights_w = np.array(count_w) / len(p_w)
gap_w = np.array(gap_w)
ece_w = float(np.sum(weights_w * gap_w))
print("bin weights:", np.round(weights_w, 3))
print("weighted gaps:", np.round(weights_w * gap_w, 3))
print("ECE:", round(ece_w, 3))
assert round(ece_w, 3) == 0.25

▶ What you'll see: the ECE is 0.250, an average absolute frequency-vs-probability mismatch across bins.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["low", "mid", "high"], weights_w * gap_w, color="darkorange")
plt.ylabel("weight × calibration gap")
plt.title("3: contributions to ECE")
plt.show()

▶ What you'll see: the low and mid bins contribute most of the total ECE because they combine nontrivial gaps with enough examples.

*Why it's done this way:* An unweighted average of gaps would let a tiny bin count as much as a large bin. Weighting by bin mass approximates the expectation over future examples: the calibration error you expect to experience depends both on how wrong a region is and how often predictions land there.

### 4. Proper losses score probability quality example by example

Calibration checks grouped frequencies, but training usually minimizes a per-example loss. Log loss is a proper scoring rule: it rewards assigning high probability to the event that actually occurs and punishes confident mistakes sharply.

In [ ]:
p_loss_w = np.array([0.20, 0.80, 0.60])
y_loss_w = np.array([0, 1, 0])
losses_w = -(y_loss_w * np.log(p_loss_w) + (1 - y_loss_w) * np.log(1 - p_loss_w))
print("per-example log losses:", np.round(losses_w, 3))
print("average log loss:", round(float(np.mean(losses_w)), 3))
assert np.allclose(np.round(losses_w, 3), [0.223, 0.223, 0.916])

▶ What you'll see: the confident wrong-ish 0.60 prediction on a negative example has much larger loss than the two well-aligned examples.

In [ ]:
grid_w = np.linspace(0.01, 0.99, 200)
loss_if_one_w = -np.log(grid_w)
loss_if_zero_w = -np.log(1 - grid_w)
plt.figure(figsize=(5, 3))
plt.plot(grid_w, loss_if_one_w, label="y=1 loss", color="teal")
plt.plot(grid_w, loss_if_zero_w, label="y=0 loss", color="crimson")
plt.xlabel("predicted probability for class 1")
plt.ylabel("log loss")
plt.title("4: confident mistakes are expensive")
plt.legend()
plt.show()

▶ What you'll see: loss explodes when a model assigns near-zero probability to the event that actually occurs.

*Why it's done this way:* A proper loss makes honest probabilities optimal in expectation. If the true event frequency is 0.7, the expected log loss is minimized by predicting 0.7, not by pretending to be more confident. This aligns training with calibrated probability reporting.

### 5. Selection uses raw fit plus cost

The lesson's arithmetic separates raw empirical fit from the method's cost. The verified toy losses are 0.224, 0.083, and 0.505, whose empirical average is 0.271. But the decision score adds a 0.090 cost, because a model with more flexibility must pay for that flexibility before being selected.

In [ ]:
losses_select_w = np.array([0.224, 0.083, 0.505])
risk_w = float(np.mean(losses_select_w))
cost_w = 0.090
score_w = risk_w + cost_w
print("empirical risk:", round(risk_w, 3))
print("cost term:", round(cost_w, 3))
print("decision score:", round(score_w, 3))
assert round(risk_w, 3) == 0.271
assert round(score_w, 3) == 0.361

▶ What you'll see: the raw average is 0.271, but the score used for selection is 0.361 after adding cost.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["raw risk", "+ cost", "decision score"], [risk_w, cost_w, score_w], color=["teal", "orange", "purple"])
plt.ylabel("score component")
plt.title("5: selection uses the full score")
plt.show()

▶ What you'll see: the final score is not the training average alone; the orange cost is part of the comparison.

*Why it's done this way:* Empirical risk estimates fit on the sample, while the cost term encodes the price of complexity, regularization, or operation. Adding the cost keeps us from selecting a model merely because it found a flattering way to reduce the raw training term.

### 6. Gaps and stabilization decide what to carry forward

A more flexible alternative has decision score 0.397. The baseline score 0.361 is lower by 0.036, but the relative gap is only about 9.1%, so the win should be read with uncertainty in mind. If a stabilizing knob reduces the baseline score by 20%, the stabilized score becomes 0.289 and is the best of the three.

In [ ]:
baseline_w = 0.361
flexible_w = 0.397
gap_select_w = flexible_w - baseline_w
relative_gap_w = gap_select_w / flexible_w
stable_w = 0.80 * baseline_w
scores_w = np.array([baseline_w, flexible_w, stable_w])
labels_w = np.array(["baseline", "flexible", "stabilized"])
print("absolute gap:", round(gap_select_w, 3))
print("relative gap:", round(relative_gap_w, 3))
print("stabilized score:", round(stable_w, 3))
print("winner:", labels_w[int(np.argmin(scores_w))])
assert round(gap_select_w, 3) == 0.036
assert round(relative_gap_w, 3) == 0.091
assert round(stable_w, 3) == 0.289

▶ What you'll see: the stabilized score is the minimum, while the baseline-vs-flexible gap is small enough to treat as evidence rather than destiny.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(labels_w, scores_w, color=["gray", "crimson", "seagreen"])
plt.ylabel("lower is better")
plt.title("6: final decision scores")
plt.show()

▶ What you'll see: the green stabilized bar is lowest, so this toy decision carries the stabilized version forward.

*Why it's done this way:* Calibration work is still model selection. We compare complete scores on the same scale, ask whether the gap is meaningful, and prefer the setting that is both accurate and stable enough to survive future data.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, masks, binning, probabilities, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for reliability diagrams, bars, curves, and calibration diagnostics.
np.random.seed(0) # make every random example reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Read probabilities as promises

**Goal.** Store predicted probabilities and labels, because calibration compares what the model promised with what actually happened. We build it in 2 steps.

In [ ]:
p_b1 = np.array([0.10, 0.30, 0.70, 0.90]) # define four model probabilities for class 1.
y_b1 = np.array([0, 0, 1, 1]) # define the observed outcomes for the same four examples.
print("probabilities:", p_b1) # inspect the probability promises.
print("labels:", y_b1) # inspect the realized binary outcomes.

▶ What you'll see: each prediction is a probability between 0 and 1, while each outcome is a hard 0 or 1.

In [ ]:
mean_p_b1 = float(np.mean(p_b1)) # average the model's promises over this tiny group.
freq_b1 = float(np.mean(y_b1)) # average labels to get the empirical event rate.
print("average predicted probability:", round(mean_p_b1, 3)) # inspect the promised frequency.
print("observed event frequency:", round(freq_b1, 3)) # inspect the realized frequency.
assert round(mean_p_b1, 3) == 0.5 # verify the tiny promise average.
assert round(freq_b1, 3) == 0.5 # verify the tiny observed frequency.

▶ What you'll see: this tiny group is perfectly calibrated on average, even though individual examples are still 0 or 1.

👀 Takeaway: calibration is about average frequencies across groups, not about making every individual label equal its probability.

### Basic 2 — Compute one bin's calibration gap

**Goal.** Compare one bin's observed frequency with its average prediction, because the calibration gap is the atomic calculation in this lesson. We build it in 2 steps.

In [ ]:
p_b2 = np.array([0.62, 0.68, 0.71, 0.79]) # collect predictions that all live in a high-probability bin.
y_b2 = np.array([1, 1, 0, 1]) # collect the matching outcomes.
print("bin probabilities:", p_b2) # inspect probabilities before averaging.
print("bin labels:", y_b2) # inspect labels before averaging.

▶ What you'll see: the bin contains four predictions near 0.70 and three positive outcomes.

In [ ]:
avg_p_b2 = float(np.mean(p_b2)) # compute avg(p in bin).
freq_b2 = float(np.mean(y_b2)) # compute Pr(Y=1 | p in bin) empirically.
gap_b2 = abs(freq_b2 - avg_p_b2) # compute the absolute calibration gap.
print("avg p:", round(avg_p_b2, 3), "frequency:", round(freq_b2, 3), "gap:", round(gap_b2, 3)) # inspect all pieces.
assert round(avg_p_b2, 3) == 0.7 # verify the average probability.
assert round(freq_b2, 3) == 0.75 # verify the event frequency.
assert round(gap_b2, 3) == 0.05 # verify the calibration gap.

▶ What you'll see: the model promised 0.700, the bin delivered 0.750, so the local gap is 0.050.

👀 Takeaway: a bin is calibrated when its empirical event rate matches its average predicted probability.

### Basic 3 — Assign predictions to bins

**Goal.** Use numeric bin edges to group predictions, because calibration is estimated from enough examples per region. We build it in 2 steps.

In [ ]:
p_b3 = np.array([0.05, 0.22, 0.48, 0.51, 0.73, 0.88]) # define probabilities spread across the unit interval.
edges_b3 = np.array([0.0, 0.33, 0.66, 1.0]) # define three bins: low, middle, and high.
ids_b3 = np.digitize(p_b3, edges_b3[1:-1], right=True) # convert probabilities into bin ids 0, 1, or 2.
print("probabilities:", p_b3) # inspect the values being binned.
print("bin ids:", ids_b3) # inspect each probability's assigned bin.

▶ What you'll see: small probabilities land in bin 0, mid probabilities in bin 1, and large probabilities in bin 2.

In [ ]:
counts_b3 = np.array([np.sum(ids_b3 == k_b3) for k_b3 in range(3)]) # count examples in each bin.
print("bin counts:", counts_b3) # inspect how much evidence each bin has.
plt.figure(figsize=(4, 3)) # create a compact histogram-style plot.
plt.bar(["low", "mid", "high"], counts_b3, color="teal") # show count per calibration bin.
plt.title("Basic 3: prediction counts per bin") # title the bin count plot.
plt.ylabel("examples") # label the count axis.
plt.show() # display the plot.

▶ What you'll see: each bin contains exactly two examples in this deliberately balanced toy setup.

👀 Takeaway: bin counts matter because a calibration frequency estimated from too few examples is noisy.

### Basic 4 — Draw a tiny reliability diagram

**Goal.** Plot observed frequency against average prediction, because the diagonal is the visual signature of perfect calibration. We build it in 2 steps.

In [ ]:
avg_p_b4 = np.array([0.15, 0.50, 0.85]) # define average predicted probabilities for three bins.
freq_b4 = np.array([0.10, 0.60, 0.80]) # define observed event frequencies for the same bins.
gaps_b4 = np.abs(freq_b4 - avg_p_b4) # compute vertical distances from the diagonal.
print("gaps:", np.round(gaps_b4, 3)) # inspect local calibration errors.
assert np.allclose(np.round(gaps_b4, 3), [0.05, 0.10, 0.05]) # verify the toy gaps.

▶ What you'll see: the middle bin is the least calibrated because its gap is 0.10.

In [ ]:
plt.figure(figsize=(4, 3)) # create a reliability diagram.
plt.plot([0, 1], [0, 1], "k--") # draw the perfect-calibration diagonal.
plt.scatter(avg_p_b4, freq_b4, s=80, color="purple") # plot each bin as one point.
plt.xlabel("average predicted probability") # label the promise axis.
plt.ylabel("observed frequency") # label the empirical frequency axis.
plt.title("Basic 4: reliability diagram") # title the plot.
plt.show() # display the diagram.

▶ What you'll see: points above the diagonal are underconfident, and points below it are overconfident.

👀 Takeaway: reliability diagrams make the sign and size of calibration errors visible.

### Basic 5 — Compute ECE by hand

**Goal.** Weight bin gaps by bin mass, because expected calibration error should reflect where predictions actually occur. We build it in 2 steps.

In [ ]:
counts_b5 = np.array([50, 30, 20]) # define how many examples fall in each bin.
gaps_b5 = np.array([0.02, 0.10, 0.05]) # define each bin's absolute calibration gap.
weights_b5 = counts_b5 / np.sum(counts_b5) # convert counts into fractions of the dataset.
print("weights:", weights_b5) # inspect bin masses.
print("gaps:", gaps_b5) # inspect local errors.

▶ What you'll see: the largest bin receives weight 0.5, so its gap matters most in the expectation.

In [ ]:
ece_b5 = float(np.sum(weights_b5 * gaps_b5)) # compute weighted average absolute calibration error.
print("ECE:", round(ece_b5, 3)) # inspect the summary error.
assert round(ece_b5, 3) == 0.05 # verify 0.5*0.02 + 0.3*0.10 + 0.2*0.05.
plt.figure(figsize=(4, 3)) # create a contribution plot.
plt.bar(["bin0", "bin1", "bin2"], weights_b5 * gaps_b5, color="orange") # show contribution from each bin.
plt.title("Basic 5: ECE contributions") # title the plot.
plt.ylabel("weight × gap") # label the contribution axis.
plt.show() # display the plot.

▶ What you'll see: bin 1 contributes the most despite not being the largest, because its gap is much bigger.

👀 Takeaway: ECE is a frequency-weighted average of absolute bin calibration gaps.

### Basic 6 — Identify overconfidence and underconfidence

**Goal.** Keep the signed gap before taking absolute value, because the sign tells us whether probabilities are too high or too low. We build it in 2 steps.

In [ ]:
avg_p_b6 = np.array([0.25, 0.55, 0.85]) # define average model probabilities by bin.
freq_b6 = np.array([0.35, 0.50, 0.70]) # define observed event rates by bin.
signed_b6 = freq_b6 - avg_p_b6 # positive means events happen more often than predicted.
print("signed gaps:", np.round(signed_b6, 3)) # inspect direction of miscalibration.

▶ What you'll see: the low bin is underconfident, while the mid and high bins are overconfident.

In [ ]:
plt.figure(figsize=(4, 3)) # create a signed calibration plot.
plt.bar(["low", "mid", "high"], signed_b6, color=["seagreen", "crimson", "crimson"]) # show signed gaps.
plt.axhline(0, color="black", linewidth=1) # add the calibrated zero-gap reference.
plt.title("Basic 6: signed calibration gap") # title the plot.
plt.ylabel("frequency - average probability") # label the signed error axis.
plt.show() # display the plot.

▶ What you'll see: bars above zero mean predictions are too small; bars below zero mean predictions are too large.

👀 Takeaway: absolute gaps summarize error size, while signed gaps diagnose direction.

### Basic 7 — Compute Brier score

**Goal.** Measure squared probability error, because the Brier score is a simple proper loss for probabilistic binary predictions. We build it in 2 steps.

In [ ]:
p_b7 = np.array([0.10, 0.70, 0.80, 0.40]) # define predicted probabilities.
y_b7 = np.array([0, 1, 0, 1]) # define observed labels.
errors_b7 = p_b7 - y_b7 # compute signed probability residuals.
print("probability residuals:", errors_b7) # inspect which predictions missed high or low.

▶ What you'll see: confident wrong predictions create large residual magnitudes.

In [ ]:
brier_terms_b7 = errors_b7 ** 2 # square residuals so positive and negative misses both count.
brier_b7 = float(np.mean(brier_terms_b7)) # average squared probability error.
print("Brier terms:", np.round(brier_terms_b7, 3)) # inspect per-example losses.
print("Brier score:", round(brier_b7, 3)) # inspect the average proper score.
assert round(brier_b7, 3) == 0.275 # verify the hand-checkable score.
plt.figure(figsize=(4, 3)) # create a small loss plot.
plt.bar(range(len(brier_terms_b7)), brier_terms_b7, color="slateblue") # show per-example Brier losses.
plt.title("Basic 7: per-example Brier terms") # title the plot.
plt.ylabel("(p - y)^2") # label the loss axis.
plt.show() # display the plot.

▶ What you'll see: the example with p=0.80 and y=0 dominates the score.

👀 Takeaway: Brier score rewards accurate probabilities and penalizes confident mistakes quadratically.

### Basic 8 — Compute log loss carefully

**Goal.** Evaluate probabilities with log loss, because log loss strongly punishes probabilities that are confidently assigned to the wrong outcome. We build it in 2 steps.

In [ ]:
p_b8 = np.array([0.20, 0.80, 0.60]) # define predicted probabilities for class 1.
y_b8 = np.array([0, 1, 0]) # define observed labels.
p_safe_b8 = np.clip(p_b8, 1e-12, 1 - 1e-12) # guard logs against exact 0 or 1.
print("safe probabilities:", p_safe_b8) # inspect clipped probabilities.

▶ What you'll see: clipping leaves these probabilities unchanged because none are exactly 0 or 1.

In [ ]:
loss_b8 = -(y_b8 * np.log(p_safe_b8) + (1 - y_b8) * np.log(1 - p_safe_b8)) # compute binary log loss terms.
mean_loss_b8 = float(np.mean(loss_b8)) # average the per-example losses.
print("log-loss terms:", np.round(loss_b8, 3)) # inspect individual penalties.
print("mean log loss:", round(mean_loss_b8, 3)) # inspect the empirical risk.
assert np.allclose(np.round(loss_b8, 3), [0.223, 0.223, 0.916]) # verify canonical terms.

▶ What you'll see: the p=0.60 prediction for a negative label costs 0.916, much more than the good predictions.

👀 Takeaway: log loss is sensitive to calibration because dishonest confidence becomes expensive.

### Basic 9 — Add a cost term to empirical risk

**Goal.** Reproduce the lesson's raw average and cost-adjusted score, because selection should use the full score. We build it in 2 steps.

In [ ]:
losses_b9 = np.array([0.224, 0.083, 0.505]) # use the verified toy per-example losses from the lesson prose.
cost_b9 = 0.090 # define the method's complexity or operational cost.
risk_b9 = float(np.mean(losses_b9)) # compute empirical risk.
print("risk:", round(risk_b9, 3), "cost:", round(cost_b9, 3)) # inspect raw fit and cost separately.
assert round(risk_b9, 3) == 0.271 # verify the average loss.

▶ What you'll see: the raw empirical risk is 0.271 before any cost is included.

In [ ]:
score_b9 = risk_b9 + cost_b9 # add cost to get the decision score.
print("decision score:", round(score_b9, 3)) # inspect the selection score.
assert round(score_b9, 3) == 0.361 # verify the lesson's decision score.
plt.figure(figsize=(4, 3)) # create a component plot.
plt.bar(["risk", "cost", "score"], [risk_b9, cost_b9, score_b9], color=["teal", "orange", "purple"]) # compare score pieces.
plt.title("Basic 9: risk plus cost") # title the plot.
plt.ylabel("value") # label the score scale.
plt.show() # display the plot.

▶ What you'll see: the total score is visibly larger than the raw risk because the cost is part of the decision.

👀 Takeaway: a calibrated modeling workflow compares full decision scores, not isolated training losses.

### Basic 10 — Pick the minimum decision score

**Goal.** Compare baseline, flexible, and stabilized scores, because the selected model is the one with the lowest valid score on the same scale. We build it in 2 steps.

In [ ]:
scores_b10 = np.array([0.361, 0.397, 0.289]) # define baseline, flexible alternative, and stabilized scores.
names_b10 = np.array(["baseline", "flexible", "stabilized"]) # name each candidate.
best_idx_b10 = int(np.argmin(scores_b10)) # find the lowest score.
print("scores:", dict(zip(names_b10, scores_b10))) # inspect all candidates.
print("best:", names_b10[best_idx_b10]) # inspect the selected candidate.
assert names_b10[best_idx_b10] == "stabilized" # verify the lesson's final winner.

▶ What you'll see: stabilized has the smallest score, so it wins this toy comparison.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create a final-score comparison plot.
plt.bar(names_b10, scores_b10, color=["gray", "crimson", "seagreen"]) # draw one bar per candidate.
plt.title("Basic 10: lower calibrated decision score wins") # title the plot.
plt.ylabel("decision score") # label the score axis.
plt.show() # display the plot.

▶ What you'll see: the green bar is lowest, matching the printed selected model.

👀 Takeaway: model choice is a same-scale minimization after fit, cost, and stabilization are all included.

## 🟡 Easy

### Easy 1 — Build calibration bins from raw predictions

**Goal.** Compute a full reliability table from predictions and labels, because calibration diagnostics start with bin-level averages. We build it in 3 steps.

In [ ]:
p_e1 = np.array([0.05, 0.15, 0.25, 0.35, 0.55, 0.65, 0.75, 0.85, 0.95]) # define nine probabilities across the range.
y_e1 = np.array([0, 0, 0, 1, 0, 1, 1, 1, 1]) # define observed binary outcomes.
edges_e1 = np.array([0.0, 0.33, 0.66, 1.0]) # define low, middle, and high bins.
ids_e1 = np.digitize(p_e1, edges_e1[1:-1], right=True) # assign each probability to a bin.
print("bin ids:", ids_e1) # inspect assignments.

▶ What you'll see: the predictions are divided into three probability regions.

In [ ]:
avg_p_e1, freq_e1, count_e1 = [], [], [] # prepare reliability-table columns.
for k_e1 in range(3): # loop through low, middle, and high bins.
    mask_e1 = ids_e1 == k_e1 # select examples in the current bin.
    avg_p_e1.append(float(np.mean(p_e1[mask_e1]))) # average predicted probability.
    freq_e1.append(float(np.mean(y_e1[mask_e1]))) # observed event frequency.
    count_e1.append(int(np.sum(mask_e1))) # number of examples in the bin.
print("counts:", count_e1) # inspect bin evidence.
print("avg p:", np.round(avg_p_e1, 3)) # inspect predicted frequencies.
print("freq:", np.round(freq_e1, 3)) # inspect observed frequencies.

▶ What you'll see: each bin reports how many examples it has, what it promised, and what happened.

In [ ]:
gap_e1 = np.abs(np.array(freq_e1) - np.array(avg_p_e1)) # compute absolute calibration gap per bin.
print("gaps:", np.round(gap_e1, 3)) # inspect local errors.
plt.figure(figsize=(4.5, 3)) # create a reliability diagram.
plt.plot([0, 1], [0, 1], "k--") # draw perfect calibration.
plt.scatter(avg_p_e1, freq_e1, s=np.array(count_e1) * 60, color="purple") # plot bins sized by count.
plt.title("Easy 1: reliability table as a plot") # title the plot.
plt.xlabel("avg predicted probability") # label x-axis.
plt.ylabel("observed frequency") # label y-axis.
plt.show() # display the diagram.

▶ What you'll see: the plotted bins reveal where the model is high or low relative to the diagonal.

👀 Takeaway: a reliability table is just bin counts, average predictions, observed frequencies, and gaps.

### Easy 2 — Compare two models by ECE

**Goal.** Compute ECE for two probability vectors on the same labels, because calibration comparison requires the same bins and same data. We build it in 3 steps.

In [ ]:
y_e2 = np.array([0, 0, 0, 1, 1, 1, 1, 0]) # define common labels for both models.
p_model1_e2 = np.array([0.05, 0.20, 0.35, 0.45, 0.65, 0.70, 0.85, 0.90]) # define model 1 probabilities.
p_model2_e2 = np.array([0.15, 0.25, 0.30, 0.55, 0.60, 0.75, 0.80, 0.70]) # define model 2 probabilities.
edges_e2 = np.array([0.0, 0.5, 0.75, 1.0]) # define shared bins.
print("labels:", y_e2) # inspect common outcomes.

▶ What you'll see: both models will be judged against the exact same binary outcomes.

In [ ]:
eces_e2 = [] # prepare one ECE per model.
for p_now_e2 in [p_model1_e2, p_model2_e2]: # loop over the two model outputs.
    ids_now_e2 = np.digitize(p_now_e2, edges_e2[1:-1], right=True) # bin current model predictions.
    ece_now_e2 = 0.0 # initialize current ECE.
    for k_e2 in range(3): # loop over bins.
        mask_e2 = ids_now_e2 == k_e2 # select current bin.
        if np.sum(mask_e2) > 0: # skip empty bins.
            ece_now_e2 += np.mean(mask_e2) * abs(float(np.mean(y_e2[mask_e2])) - float(np.mean(p_now_e2[mask_e2]))) # add weighted gap.
    eces_e2.append(ece_now_e2) # store current model ECE.
print("ECEs:", np.round(eces_e2, 3)) # inspect calibration summaries.

▶ What you'll see: model 2 has the lower ECE on these bins, so it is better calibrated by this diagnostic.

In [ ]:
plt.figure(figsize=(4, 3)) # create an ECE comparison plot.
plt.bar(["model 1", "model 2"], eces_e2, color=["crimson", "seagreen"]) # compare ECE values.
plt.title("Easy 2: lower ECE is better") # title the plot.
plt.ylabel("ECE") # label the error axis.
plt.show() # display the plot.

▶ What you'll see: the shorter bar identifies the better-calibrated model under these bins.

👀 Takeaway: calibration metrics must be compared with the same data, binning rule, and scale.

### Easy 3 — Apply a simple shrinkage calibration

**Goal.** Pull extreme probabilities toward the base rate, because stabilization can reduce brittle overconfidence. We build it in 3 steps.

In [ ]:
p_e3 = np.array([0.02, 0.10, 0.20, 0.80, 0.90, 0.98]) # define overconfident probabilities near 0 and 1.
y_e3 = np.array([0, 0, 1, 1, 1, 0]) # define labels with one confident high-probability miss.
base_e3 = float(np.mean(y_e3)) # compute the empirical base rate.
alpha_e3 = 0.25 # set shrinkage strength toward the base rate.
print("base rate:", round(base_e3, 3), "alpha:", alpha_e3) # inspect calibration ingredients.

▶ What you'll see: the base rate is 0.500, so shrinkage pulls probabilities toward the center.

In [ ]:
p_shrunk_e3 = (1 - alpha_e3) * p_e3 + alpha_e3 * base_e3 # shrink every probability toward the base rate.
print("original:", np.round(p_e3, 3)) # inspect original probabilities.
print("shrunk:  ", np.round(p_shrunk_e3, 3)) # inspect stabilized probabilities.
assert round(float(p_shrunk_e3[0]), 3) == 0.14 # verify the first shrunken probability.

▶ What you'll see: 0.02 becomes 0.14 and 0.98 becomes 0.86, so extremes are softened.

In [ ]:
brier_orig_e3 = float(np.mean((p_e3 - y_e3) ** 2)) # compute original Brier score.
brier_shrunk_e3 = float(np.mean((p_shrunk_e3 - y_e3) ** 2)) # compute shrunken Brier score.
print("Brier original:", round(brier_orig_e3, 3), "shrunk:", round(brier_shrunk_e3, 3)) # compare probability quality.
plt.figure(figsize=(5, 3)) # create a probability comparison plot.
plt.plot(p_e3, marker="o", label="original") # plot original probabilities.
plt.plot(p_shrunk_e3, marker="s", label="shrunk") # plot stabilized probabilities.
plt.title("Easy 3: shrink probabilities toward base rate") # title the plot.
plt.ylabel("probability") # label probability axis.
plt.legend() # show line labels.
plt.show() # display the plot.

▶ What you'll see: shrinkage reduces extreme confidence and may improve or hurt the proper score depending on the sample.

👀 Takeaway: stabilization is a bias-variance tradeoff for probabilities, not a guaranteed free improvement.

### Easy 4 — Choose a decision threshold from calibrated probabilities

**Goal.** Convert probabilities into actions only after calibration, because a threshold has a frequency interpretation when probabilities are honest. We build it in 3 steps.

In [ ]:
p_e4 = np.array([0.12, 0.28, 0.44, 0.57, 0.71, 0.86]) # define calibrated-ish risk scores.
y_e4 = np.array([0, 0, 0, 1, 1, 1]) # define observed outcomes for checking.
thresholds_e4 = np.array([0.3, 0.5, 0.7]) # define candidate action thresholds.
print("thresholds:", thresholds_e4) # inspect threshold grid.

▶ What you'll see: the same probabilities will produce different actions depending on the chosen threshold.

In [ ]:
positive_rates_e4 = [] # store fraction flagged positive at each threshold.
precisions_e4 = [] # store empirical precision at each threshold.
for t_e4 in thresholds_e4: # loop over thresholds.
    pred_e4 = p_e4 >= t_e4 # take action when probability exceeds threshold.
    positive_rates_e4.append(float(np.mean(pred_e4))) # compute action rate.
    precisions_e4.append(float(np.mean(y_e4[pred_e4])) if np.any(pred_e4) else 0.0) # compute observed hit rate among flagged cases.
print("action rates:", np.round(positive_rates_e4, 3)) # inspect how many cases are selected.
print("precision:", np.round(precisions_e4, 3)) # inspect observed success among selected cases.

▶ What you'll see: higher thresholds select fewer cases but with higher empirical precision in this monotone toy example.

In [ ]:
plt.figure(figsize=(5, 3)) # create a threshold tradeoff plot.
plt.plot(thresholds_e4, positive_rates_e4, marker="o", label="action rate") # plot selected fraction.
plt.plot(thresholds_e4, precisions_e4, marker="s", label="precision") # plot hit rate among selected cases.
plt.title("Easy 4: threshold tradeoff") # title the plot.
plt.xlabel("threshold") # label threshold axis.
plt.legend() # show curve labels.
plt.show() # display the plot.

▶ What you'll see: threshold choice changes operational behavior even when probabilities stay fixed.

👀 Takeaway: calibrated probabilities support threshold decisions because their scale has a frequency meaning.

### Easy 5 — Reproduce the lesson's gap arithmetic

**Goal.** Compute absolute gap, relative gap, and stabilized score, because the lesson's final comparison depends on all three quantities. We build it in 3 steps.

In [ ]:
baseline_e5 = 0.361 # define the baseline decision score.
flexible_e5 = 0.397 # define the more flexible alternative's score.
stabilizing_fraction_e5 = 0.80 # define the 20% reduction from the stabilizing knob.
print("baseline:", baseline_e5, "flexible:", flexible_e5) # inspect candidate scores.

▶ What you'll see: the flexible model is worse here because its score is higher.

In [ ]:
gap_e5 = flexible_e5 - baseline_e5 # compute absolute evidence gap.
relative_gap_e5 = gap_e5 / flexible_e5 # express the gap relative to the alternative.
stable_e5 = stabilizing_fraction_e5 * baseline_e5 # compute the stabilized score.
print("gap:", round(gap_e5, 3)) # inspect absolute score difference.
print("relative gap:", round(relative_gap_e5, 3)) # inspect scale-aware score difference.
print("stabilized:", round(stable_e5, 3)) # inspect the new stable score.
assert round(gap_e5, 3) == 0.036 # verify lesson gap.
assert round(relative_gap_e5, 3) == 0.091 # verify lesson relative gap.
assert round(stable_e5, 3) == 0.289 # verify lesson stabilized score.

▶ What you'll see: the baseline beats flexible by 0.036, and stabilization lowers the score to 0.289.

In [ ]:
scores_e5 = np.array([baseline_e5, flexible_e5, stable_e5]) # gather all decision scores.
labels_e5 = np.array(["baseline", "flexible", "stable"]) # name each score.
print("winner:", labels_e5[int(np.argmin(scores_e5))]) # inspect final selected setting.
plt.figure(figsize=(4.5, 3)) # create final comparison plot.
plt.bar(labels_e5, scores_e5, color=["gray", "crimson", "seagreen"]) # compare same-scale scores.
plt.title("Easy 5: lesson decision arithmetic") # title the plot.
plt.ylabel("score") # label the score axis.
plt.show() # display the plot.

▶ What you'll see: the stable setting has the lowest bar and is selected.

👀 Takeaway: the full comparison is baseline 0.361, flexible 0.397, and stabilized 0.289.

## 🔴 Advanced

### Advanced 1 — Bootstrap calibration uncertainty

**Goal.** Estimate how noisy ECE can be on a small validation set, because a tiny calibration improvement can disappear under resampling. We build it in 4 steps.

In [ ]:
p_a1 = np.array([0.05, 0.12, 0.20, 0.35, 0.48, 0.60, 0.72, 0.85, 0.92, 0.97]) # define validation probabilities.
y_a1 = np.array([0, 0, 0, 0, 1, 1, 0, 1, 1, 1]) # define validation labels.
edges_a1 = np.array([0.0, 0.33, 0.66, 1.0]) # define fixed ECE bins.
rng_a1 = np.random.default_rng(1) # create reproducible bootstrap randomness.
print("validation size:", len(p_a1)) # inspect sample size.

▶ What you'll see: only ten validation examples are available, so uncertainty matters.

In [ ]:
def ece_three_bins_a1(p_now_a1, y_now_a1): # define a local ECE helper for this advanced example.
    ids_now_a1 = np.digitize(p_now_a1, edges_a1[1:-1], right=True) # bin probabilities.
    total_now_a1 = 0.0 # initialize ECE.
    for k_now_a1 in range(3): # loop through bins.
        mask_now_a1 = ids_now_a1 == k_now_a1 # select bin examples.
        if np.sum(mask_now_a1) > 0: # skip empty bins.
            total_now_a1 += np.mean(mask_now_a1) * abs(float(np.mean(y_now_a1[mask_now_a1])) - float(np.mean(p_now_a1[mask_now_a1]))) # add weighted gap.
    return total_now_a1 # return ECE.
base_ece_a1 = ece_three_bins_a1(p_a1, y_a1) # compute original validation ECE.
print("original ECE:", round(base_ece_a1, 3)) # inspect point estimate.

▶ What you'll see: the validation set has one ECE number, but it is only a sample estimate.

In [ ]:
boot_a1 = [] # store bootstrap ECE values.
for rep_a1 in range(300): # run many resamples.
    idx_a1 = rng_a1.integers(0, len(p_a1), len(p_a1)) # sample validation rows with replacement.
    boot_a1.append(ece_three_bins_a1(p_a1[idx_a1], y_a1[idx_a1])) # compute ECE on the resample.
boot_a1 = np.array(boot_a1) # convert results to an array.
lo_a1, hi_a1 = np.percentile(boot_a1, [5, 95]) # compute a central bootstrap interval.
print("5th percentile:", round(float(lo_a1), 3), "95th percentile:", round(float(hi_a1), 3)) # inspect uncertainty interval.

▶ What you'll see: the bootstrap interval is wide enough that small ECE differences should be interpreted cautiously.

In [ ]:
plt.figure(figsize=(5, 3)) # create a bootstrap histogram.
plt.hist(boot_a1, bins=18, color="steelblue", edgecolor="white") # show resampled ECE distribution.
plt.axvline(base_ece_a1, color="black", linestyle="--", label="original") # mark original estimate.
plt.title("Advanced 1: bootstrap ECE uncertainty") # title the plot.
plt.xlabel("ECE") # label ECE axis.
plt.legend() # show original estimate label.
plt.show() # display the histogram.

▶ What you'll see: ECE varies across resamples, so the validation gap is an evidence distribution, not a fixed truth.

👀 Takeaway: calibration improvements should be larger than validation noise before we trust them.

### Advanced 2 — Tune shrinkage on validation data

**Goal.** Select a shrinkage strength using validation Brier score, because stabilization should be chosen by future-facing evidence. We build it in 4 steps.

In [ ]:
p_a2 = np.array([0.02, 0.08, 0.18, 0.33, 0.57, 0.69, 0.81, 0.93]) # define uncalibrated probabilities.
y_a2 = np.array([0, 0, 1, 0, 1, 1, 1, 0]) # define validation labels.
base_a2 = float(np.mean(y_a2)) # compute validation base rate for shrinkage target.
alphas_a2 = np.array([0.0, 0.1, 0.25, 0.5, 0.75]) # define shrinkage strengths to try.
print("base rate:", round(base_a2, 3)) # inspect shrinkage target.

▶ What you'll see: probabilities will be pulled toward the validation base rate as alpha increases.

In [ ]:
briers_a2 = [] # store Brier score for each alpha.
for alpha_a2 in alphas_a2: # loop over shrinkage strengths.
    p_cal_a2 = (1 - alpha_a2) * p_a2 + alpha_a2 * base_a2 # shrink probabilities toward base rate.
    briers_a2.append(float(np.mean((p_cal_a2 - y_a2) ** 2))) # compute validation Brier score.
print("alphas:", alphas_a2) # inspect candidate strengths.
print("Brier scores:", np.round(briers_a2, 3)) # inspect validation losses.

▶ What you'll see: the validation curve chooses the amount of stabilization rather than assuming more shrinkage is always better.

In [ ]:
best_idx_a2 = int(np.argmin(briers_a2)) # find best validation Brier score.
best_alpha_a2 = float(alphas_a2[best_idx_a2]) # read best alpha.
print("best alpha:", best_alpha_a2) # inspect selected shrinkage.
assert best_alpha_a2 in alphas_a2 # verify the selected value comes from the tested grid.

▶ What you'll see: one alpha has the lowest validation Brier score on this sample.

In [ ]:
plt.figure(figsize=(5, 3)) # create a tuning curve plot.
plt.plot(alphas_a2, briers_a2, marker="o", color="purple") # draw validation Brier versus shrinkage.
plt.axvline(best_alpha_a2, color="red", linestyle="--", label="best alpha") # mark selected alpha.
plt.title("Advanced 2: tune shrinkage by validation score") # title the plot.
plt.xlabel("shrinkage alpha") # label alpha axis.
plt.ylabel("validation Brier score") # label loss axis.
plt.legend() # show best-alpha marker.
plt.show() # display the tuning curve.

▶ What you'll see: the lowest point marks the chosen stabilization strength.

👀 Takeaway: a calibration knob is a hyperparameter and should be selected on validation data.

### Advanced 3 — Compare calibration and discrimination

**Goal.** Show that ranking quality and calibration quality are different, because a model can order examples well while its probabilities are too extreme. We build it in 4 steps.

In [ ]:
y_a3 = np.array([0, 0, 0, 1, 1, 1]) # define labels sorted from negatives to positives.
p_good_rank_a3 = np.array([0.05, 0.10, 0.20, 0.80, 0.90, 0.95]) # define extreme but well-ranked probabilities.
p_soft_a3 = np.array([0.25, 0.35, 0.45, 0.55, 0.65, 0.75]) # define softer probabilities with the same ranking.
print("rank order extreme:", np.argsort(p_good_rank_a3)) # inspect ranking from low to high.
print("rank order soft:", np.argsort(p_soft_a3)) # inspect ranking from low to high.

▶ What you'll see: both models rank examples in the same order.

In [ ]:
brier_extreme_a3 = float(np.mean((p_good_rank_a3 - y_a3) ** 2)) # compute Brier for extreme probabilities.
brier_soft_a3 = float(np.mean((p_soft_a3 - y_a3) ** 2)) # compute Brier for softer probabilities.
print("Brier extreme:", round(brier_extreme_a3, 3), "Brier soft:", round(brier_soft_a3, 3)) # compare probability quality.

▶ What you'll see: the extreme model has lower Brier here because its confidence matches the clean labels.

In [ ]:
y_noisy_a3 = np.array([0, 0, 1, 0, 1, 1]) # introduce a noisy inversion while keeping probabilities fixed.
brier_extreme_noisy_a3 = float(np.mean((p_good_rank_a3 - y_noisy_a3) ** 2)) # recompute extreme Brier under noise.
brier_soft_noisy_a3 = float(np.mean((p_soft_a3 - y_noisy_a3) ** 2)) # recompute soft Brier under noise.
print("noisy Brier extreme:", round(brier_extreme_noisy_a3, 3), "soft:", round(brier_soft_noisy_a3, 3)) # inspect robustness to noise.

▶ What you'll see: after a label inversion, the softer model can become preferable because extreme confidence is fragile.

In [ ]:
plt.figure(figsize=(5, 3)) # create a comparison plot.
plt.plot(p_good_rank_a3, marker="o", label="extreme") # plot extreme probabilities.
plt.plot(p_soft_a3, marker="s", label="soft") # plot softer probabilities.
plt.scatter(np.arange(len(y_noisy_a3)), y_noisy_a3, color="black", zorder=3, label="noisy labels") # overlay labels.
plt.title("Advanced 3: ranking is not calibration") # title the plot.
plt.ylabel("probability / label") # label y-axis.
plt.legend() # show line labels.
plt.show() # display the plot.

▶ What you'll see: both lines rank examples similarly, but their probability scales behave differently under noise.

👀 Takeaway: discrimination asks whether positives rank above negatives; calibration asks whether probability levels are honest frequencies.

### Advanced 4 — Penalize small-bin overinterpretation

**Goal.** Apply a simple count penalty to bin gaps, because bins with few examples can produce unstable calibration claims. We build it in 4 steps.

In [ ]:
counts_a4 = np.array([80, 15, 5]) # define uneven bin counts.
avg_p_a4 = np.array([0.20, 0.55, 0.90]) # define average predictions by bin.
freq_a4 = np.array([0.25, 0.40, 0.40]) # define observed frequencies by bin.
gaps_a4 = np.abs(freq_a4 - avg_p_a4) # compute raw calibration gaps.
print("raw gaps:", np.round(gaps_a4, 3)) # inspect unpenalized gaps.

▶ What you'll see: the tiny high-probability bin appears to have a huge raw calibration gap.

In [ ]:
standard_error_a4 = np.sqrt(freq_a4 * (1 - freq_a4) / counts_a4) # approximate frequency uncertainty per bin.
adjusted_gap_a4 = np.maximum(0, gaps_a4 - 2 * standard_error_a4) # keep only gap beyond a rough two-SE noise band.
print("standard errors:", np.round(standard_error_a4, 3)) # inspect uncertainty by bin.
print("adjusted gaps:", np.round(adjusted_gap_a4, 3)) # inspect conservative gaps.

▶ What you'll see: the small bin receives a much larger uncertainty allowance because five examples are not much evidence.

In [ ]:
weighted_raw_a4 = float(np.sum((counts_a4 / np.sum(counts_a4)) * gaps_a4)) # compute raw ECE-like summary.
weighted_adj_a4 = float(np.sum((counts_a4 / np.sum(counts_a4)) * adjusted_gap_a4)) # compute uncertainty-adjusted summary.
print("raw weighted gap:", round(weighted_raw_a4, 3)) # inspect raw summary.
print("adjusted weighted gap:", round(weighted_adj_a4, 3)) # inspect conservative summary.

▶ What you'll see: the adjusted summary is lower because some apparent gap can plausibly be sampling noise.

In [ ]:
x_a4 = np.arange(3) # create positions for grouped bars.
plt.figure(figsize=(5, 3)) # create grouped bar plot.
plt.bar(x_a4 - 0.18, gaps_a4, width=0.36, label="raw gap", color="crimson") # plot raw gaps.
plt.bar(x_a4 + 0.18, adjusted_gap_a4, width=0.36, label="adjusted gap", color="seagreen") # plot adjusted gaps.
plt.xticks(x_a4, ["low", "mid", "high"]) # label bins.
plt.title("Advanced 4: small-bin uncertainty") # title the plot.
plt.legend() # show bar labels.
plt.show() # display the plot.

▶ What you'll see: the adjusted high-bin bar shrinks because its evidence count is tiny.

👀 Takeaway: bin gaps are estimates, so count-aware uncertainty prevents overreading small validation slices.

### Advanced 5 — End-to-end calibrated model decision

**Goal.** Combine empirical risk, calibration cost, validation gap, and stabilization into one same-scale decision, because the notebook should mirror the lesson's selection logic. We build it in 4 steps.

In [ ]:
losses_base_a5 = np.array([0.224, 0.083, 0.505]) # define verified baseline losses.
losses_flex_a5 = np.array([0.190, 0.070, 0.451]) # define a tempting flexible model with lower raw losses.
cost_base_a5 = 0.090 # define baseline cost.
cost_flex_a5 = 0.160 # define larger flexible-model cost.
print("base losses:", losses_base_a5) # inspect baseline fit terms.
print("flex losses:", losses_flex_a5) # inspect flexible fit terms.

▶ What you'll see: the flexible model looks better if you only inspect raw per-example losses.

In [ ]:
risk_base_a5 = float(np.mean(losses_base_a5)) # compute baseline empirical risk.
risk_flex_a5 = float(np.mean(losses_flex_a5)) # compute flexible empirical risk.
score_base_a5 = risk_base_a5 + cost_base_a5 # compute full baseline score.
score_flex_a5 = risk_flex_a5 + cost_flex_a5 # compute full flexible score.
print("risks:", round(risk_base_a5, 3), round(risk_flex_a5, 3)) # inspect raw fit comparison.
print("scores:", round(score_base_a5, 3), round(score_flex_a5, 3)) # inspect cost-adjusted comparison.
assert round(score_base_a5, 3) == 0.361 # verify baseline score.

▶ What you'll see: the flexible model's raw risk is lower, but its cost can erase that advantage.

In [ ]:
score_stable_a5 = 0.80 * score_base_a5 # apply the lesson's 20% stabilization reduction to baseline.
all_scores_a5 = np.array([score_base_a5, score_flex_a5, score_stable_a5]) # collect comparable scores.
labels_a5 = np.array(["base", "flex", "stable"] ) # name candidates.
print("stable score:", round(score_stable_a5, 3)) # inspect stabilized result.
print("winner:", labels_a5[int(np.argmin(all_scores_a5))]) # inspect final selected candidate.
assert round(score_stable_a5, 3) == 0.289 # verify stabilized lesson score.

▶ What you'll see: stabilization produces the lowest final score in this toy decision.

In [ ]:
plt.figure(figsize=(5, 3)) # create final decision plot.
plt.bar(labels_a5, all_scores_a5, color=["gray", "crimson", "seagreen"]) # compare same-scale candidates.
plt.axhline(score_base_a5, color="gray", linestyle="--", label="baseline") # mark baseline reference.
plt.title("Advanced 5: full calibrated decision") # title the plot.
plt.ylabel("decision score") # label score axis.
plt.legend() # show baseline reference.
plt.show() # display the plot.

▶ What you'll see: the model with the best raw fit is not necessarily the one carried forward after cost and stabilization.

👀 Takeaway: probability calibration belongs inside the full validation-and-selection loop, not beside it as a decorative metric.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Calibration asks whether predicted probabilities match observed frequencies.

A calibrated probability means that examples receiving about 0.7 probability happen about 70 percent of the time. This notebook bins predicted confidence and tracks expected calibration error across harder datasets. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(7)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    cancer = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", cancer.data, cancer.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    model = LogisticRegression(max_iter=2000)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def split_scaled(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=3, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def fit_logistic(x_tr, y_tr, C=1.0):
    model = LogisticRegression(C=C, max_iter=2500)
    model.fit(x_tr, y_tr)
    return model


def classification_error(y_true, y_pred):
    return float(1.0 - accuracy_score(y_true, y_pred))


def lesson_score(losses, cost, alternative):
    raw = round(float(np.mean(losses)), 3)
    score = round(raw + cost, 3)
    gap = round(alternative - score, 3)
    relative_gap = gap / alternative
    stabilized = 0.8 * score
    return {
        "raw": raw,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
        "relative_gap": relative_gap,
        "stabilized": stabilized,
    }


def preview_ladder(rungs):
    for name, X, y in rungs:
        labels, counts = np.unique(y, return_counts=True)
        info = dict(zip(labels.tolist(), counts.tolist()))
        print(f"{name}: X={X.shape}, classes={info}, sample={np.round(X[:2], 3).tolist()}")


## The concept, built once on D1

The lesson formula is $$calibration\ gap=|\Pr(Y=1\mid \hat p\in bin)-\operatorname{avg}(\hat p\in bin)|$$. We first rebuild the exact lesson arithmetic before using a reusable method on larger data.

In [ ]:

def probability_calibration_method(losses, cost, alternative):
    """Recompute the lesson raw average, cost-aware score, and alternative gap."""
    losses = np.asarray(losses, dtype=float)
    summary = lesson_score(losses, cost, alternative)
    return summary


losses = np.array([0.224, 0.083, 0.505], dtype=float)
summary = probability_calibration_method(losses, cost=0.090, alternative=0.397)

assert abs(summary["raw"] - 0.271) < 1e-12
assert abs(summary["score"] - 0.361) < 1e-12
assert abs(summary["gap"] - 0.036) < 1e-12
assert abs(summary["relative_gap"] - 0.091) < 0.001
assert abs(summary["stabilized"] - 0.289) < 0.001

print("losses:", losses.tolist())
print("raw average:", round(summary["raw"], 3))
print("score with cost:", round(summary["score"], 3))
print("gap to alternative:", round(summary["gap"], 3))
print("relative gap:", round(summary["relative_gap"], 3))


The same score object also carries the stabilized launch number and, for this topic, the topic-specific metric calculation used on the toy D1 counts or residuals.

In [ ]:

def expected_calibration_error(probs, y_true, n_bins=3):
    probs = np.asarray(probs, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    total = len(probs)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (probs >= lo) & (probs <= hi if hi == 1.0 else probs < hi)
        if np.any(mask):
            observed = float(np.mean(y_true[mask]))
            predicted = float(np.mean(probs[mask]))
            ece += float(np.mean(mask)) * abs(observed - predicted)
    return ece

toy_probs = np.array([0.10, 0.20, 0.35, 0.65, 0.80, 0.90])
toy_y = np.array([0, 0, 1, 1, 1, 1])
toy_ece = expected_calibration_error(toy_probs, toy_y, n_bins=3)
assert round(toy_ece, 3) == 0.267


print("toy ECE:", round(toy_ece, 3))
print("first-bin gap:", round(abs(0.0 - 0.15), 3))


## The dataset ladder

The notebook embeds the shared `clf_ladder()` source so it is self-contained in Colab. D1 is inspectable; D5 is real, higher-dimensional, and imbalanced enough to make the checks matter.

In [ ]:

rungs = clf_ladder()
preview_ladder(rungs)


## Run calibration error across D1–D5

For multiclass rungs, confidence is the maximum predicted probability and correctness is whether the predicted label is right.

In [ ]:

def multiclass_ece(model, x_te, y_te, n_bins=8):
    probs = model.predict_proba(x_te)
    confidence = np.max(probs, axis=1)
    correctness = (model.predict(x_te) == y_te).astype(float)
    return expected_calibration_error(confidence, correctness, n_bins=n_bins)

results = []
for name, X, y in rungs:
    x_tr, x_te, y_tr, y_te = split_scaled(X, y)
    model = fit_logistic(x_tr, y_tr)
    ece = multiclass_ece(model, x_te, y_te)
    acc = clf_accuracy(logistic_baseline, X, y)
    results.append({"name": name, "ece": ece, "accuracy": acc})
    print(f"{name}: ECE={ece:.3f}, clf_accuracy={acc:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, (name, X, y), row in zip(axes[0], rungs, results):
    x_tr, x_te, y_tr, y_te = split_scaled(X, y)
    model = fit_logistic(x_tr, y_tr)
    probs = model.predict_proba(x_te)
    confidence = np.max(probs, axis=1)
    correctness = (model.predict(x_te) == y_te).astype(float)
    bins = np.linspace(0.0, 1.0, 6)
    centers = []
    observed = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidence >= lo) & (confidence <= hi if hi == 1.0 else confidence < hi)
        if np.any(mask):
            centers.append(float(np.mean(confidence[mask])))
            observed.append(float(np.mean(correctness[mask])))
    ax.plot([0, 1], [0, 1], color="gray", linewidth=1)
    ax.scatter(centers, observed, color="darkorange")
    ax.set_title(name.split(" (")[0], fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
axes[1, 0].plot(range(1, 6), [row["ece"] for row in results], marker="o")
axes[1, 0].set_title("ECE vs rung")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("ECE")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5: optimizing the raw term and forgetting the cost

A confident model may look attractive on raw validation error while becoming less calibrated. The cost-aware score keeps confidence on the same decision scale.

In [ ]:

name, X, y = rungs[-1]
x_tr, x_te, y_tr, y_te = split_scaled(X, y)
rows = []
for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    model = fit_logistic(x_tr, y_tr, C=C)
    preds = model.predict(x_te)
    raw_loss = classification_error(y_te, preds)
    complexity_cost = 0.090 * math.log10(C + 1.0) / math.log10(11.0)
    decision_score = raw_loss + complexity_cost
    rows.append((C, raw_loss, complexity_cost, decision_score))

raw_best = min(rows, key=lambda row: row[1])
fixed_best = min(rows, key=lambda row: row[3])
print("D5 candidate table: C, raw validation loss, cost, decision score")
for row in rows:
    print(tuple(round(value, 4) for value in row))
print("wrong raw-only choice:", raw_best)
print("cost-aware choice:", fixed_best)
print("decision-score improvement:", round(raw_best[3] - fixed_best[3], 4))
assert fixed_best[3] <= raw_best[3] + 1e-12


## Evaluate it + Practice

- Track `ECE` against a no-skill baseline before trusting the result.
- Sanity-check that D1 reproduces the lesson numbers exactly and that D5 is not a toy shortcut.
- Ablation: select the most confident model without calibration bins; the metric should get worse or the selected model should become less stable.
- Failure signals: large validation gap, unstable threshold/criterion, or a cost-aware score that disagrees with the raw metric.
- Report both the raw metric and the cost-aware decision score when the lesson formula includes a cost.

Practice prompts:
1. Change one candidate hyperparameter and recompute the D5 table.
2. Replace the no-skill baseline and explain whether `ECE` moved for the right reason.
3. Add one diagnostic plot that would catch the named pitfall for probability calibration.